In [1]:
import pandas as pd
import time

# requirements: pip install pulp

In [2]:
start=time.perf_counter()

In [3]:
#add this info to a settings dict?
current_gw = 11 #predictions from gw+1
current_season = 20262027
divx_team_code = 5049406 # My 26/27 divx team code

In [4]:
from DatabaseCreator import DatabaseCreator
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

In [5]:
from FPL_League_Data_Scraper import FPL_League_Data_Scraper

league_code = "1081956"

#Run Data Scraper for DivX league
if __name__ == "__main__":
    divx = FPL_League_Data_Scraper(league_code, current_season)
    divx_picksdf, membergw_summary = divx.scrape_picks()

#Add a new staging table & data into the database
divx_picks = divx_picksdf[['member', 'memberid','season', 'gw', 'pick', 
                 'element', 'position', 'multiplier', 
                 'is_captain', 'is_vice_captain']]

table_name = f"divx_picks_{current_season}"
if __name__ == "__main__":
    db_creator.create_staging_table_then_insert_data(table_name, data=divx_picks)
    divx_picks = db_creator.table_to_df(table_name=table_name)
    print(divx_picks.info())

Data fetched, league name:  DIV X
Fetched history for Stephen Collier (ID: 391266)
Fetched history for Joe Meader (ID: 2881244)
Fetched history for Christopher Harris (ID: 5049406)
Fetched history for Marcin Wichlinski (ID: 4010004)
Fetched history for Ollie Carroll (ID: 4913097)
Fetched history for Christopher Collier (ID: 4705527)
Fetched history for Bernard Chalk (ID: 4184853)
Fetched history for Jack Carroll (ID: 4355821)
Fetched history for Callum Harling (ID: 3081288)
Fetched history for Luiton Fernandes (ID: 3060165)
Fetched history for Anthony Collier (ID: 5037473)
Fetched history for Christopher Harris (ID: 4486919)
Fetched history for Tommy Davies (ID: 3340544)
Fetched history for Samia Shelim (ID: 678874)
Fetched history for Ryan Boyce (ID: 4036890)
Fetched history for Conner Harling (ID: 4484147)
Fetched history for Tajwar Shelim (ID: 4512122)
Fetched history for Omari Douglas-Rankine (ID: 8921685)
Fetched history for Christakis Christodoulou (ID: 5691733)
Max gameweek data

In [6]:
view = db_creator.run_sql(f"""
    WITH member_points AS (
        SELECT 
            member,
            memberid,
            SUM(
                COALESCE(playergw.total_points, 0) * COALESCE(divx_picks_{current_season}.multiplier, 0)) AS total_points   
        FROM divx_picks_{current_season}
        LEFT JOIN playergw 
            ON divx_picks_{current_season}.element = playergw.element
            AND divx_picks_{current_season}.season = playergw.season
            AND divx_picks_{current_season}.gw = playergw.event
        GROUP BY member, memberid
    )
    SELECT 
        member,
        memberid,
        total_points 
    FROM member_points
    ORDER BY total_points DESC
""")
view

,member,memberid,total_points
0,Stephen Collier,391266.0,736.0
1,Joe Meader,2881244.0,720.0
2,Christopher Harris,5049406.0,705.0
3,Marcin Wichlinski,4010004.0,685.0
4,Ollie Carroll,4913097.0,683.0
5,Christopher Collier,4705527.0,668.0
6,Bernard Chalk,4184853.0,666.0
7,Jack Carroll,4355821.0,646.0
8,Callum Harling,3081288.0,644.0
9,Luiton Fernandes,3060165.0,636.0


In [7]:
print(f"Best players for next 5 gameweeks with exponential decay (normalized): GW{current_gw+1} to GW{current_gw+5}")

# For weights to sum to 1 with decay_factor d and initial weight w:
# w * (1 + d + d^2 + d^3 + d^4) = 1
# So w = 1 / (1 + d + d^2 + d^3 + d^4)

decay_factor = 0.75 #  lower decay factor means more emphasis on nearer gws, 1 is equal weights.
weight_sum = sum([decay_factor**i for i in range(5)])
initial_weight = 1 / weight_sum # weights add to 1

view = db_creator.run_sql(f"""
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        player_name_id,
        status,
        in_my_team,
        position,
        value,
        team_name,
        ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
        ROUND(SUM(
            CASE 
                WHEN event = {current_gw+1} THEN xpoints * {initial_weight}
                WHEN event = {current_gw+2} THEN xpoints * {initial_weight * decay_factor}
                WHEN event = {current_gw+3} THEN xpoints * {initial_weight * decay_factor**2}
                WHEN event = {current_gw+4} THEN xpoints * {initial_weight * decay_factor**3}
                WHEN event = {current_gw+5} THEN xpoints * {initial_weight * decay_factor**4}
                ELSE 0
            END
        ), 2) AS total_xpoints_decayed,
        ROUND(AVG(mins_pred),1) AS avg_mins_pred,
        ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
        ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
        ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
    FROM predictions_status
    WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
        AND status IN ('a', 'd')
        --AND position IN ('DEF')
        --AND VALUE <= 44
    GROUP BY player_name_id, status, position, value, team_name, in_my_team
    ORDER BY total_xpoints_decayed DESC
""")
view.head(50)

Best players for next 5 gameweeks with exponential decay (normalized): GW12 to GW16


,player_name_id,status,in_my_team,position,value,team_name,avg_opp_team_elo,total_xpoints_decayed,avg_mins_pred,avg_xgp90_pred,avg_xap90_pred,avg_xgcp90_pred
0,Erling Haaland,a,1,FWD,158.0,Man City,1098.1,5.68,78.1,0.836,0.203,1.448
1,Pascal Groß,a,0,MID,63.0,Brighton,1049.5,5.56,84.9,0.290,0.367,1.360
2,Bruno Borges Fernandes,a,0,MID,116.0,Man Utd,1076.6,5.49,74.5,0.393,0.396,1.581
3,Morgan Gibbs-White,a,1,MID,78.0,Nott'm Forest,1059.2,5.37,85.6,0.301,0.307,1.391
4,Martin Ødegaard,a,0,MID,67.0,Arsenal,1102.6,5.32,82.3,0.347,0.251,1.087
5,Florian Wirtz,a,0,MID,73.0,Liverpool,1042.9,5.25,81.2,0.291,0.235,1.287
6,Nicolas Jackson,a,0,FWD,72.0,Aston Villa,1020.0,5.05,80.4,0.624,0.072,1.158
7,Marcus Tavernier,a,1,MID,67.0,Bournemouth,1089.8,4.96,84.7,0.249,0.254,1.398
8,Alexander Isak,a,0,FWD,93.0,Liverpool,1042.9,4.95,83.0,0.614,0.051,1.161
9,Gabriel dos Santos Magalhães,a,0,DEF,81.0,Arsenal,1102.6,4.93,84.1,0.096,0.057,1.047


In [8]:
mydivx_team_20262027 = 5049406
mydiv1_team_20262027 = 4486919

alpha = 0.5
# blending factor α (alpha) between 0 and 1, where:
# α = 0 → pure ownership-weighted (current behaviour)
# α = 1 → pure absolute points
# α = 0.5 → equal blend

ownership_weighted_predictions = db_creator.run_sql(f"""
    WITH member_points AS (
        SELECT 
            member,
            memberid,
            SUM(COALESCE(playergw.total_points, 0) * COALESCE(divx_picks_{current_season}.multiplier, 0)) AS total_points   
        FROM divx_picks_{current_season}
        LEFT JOIN playergw 
            ON divx_picks_{current_season}.element = playergw.element
            AND divx_picks_{current_season}.season = playergw.season
            AND divx_picks_{current_season}.gw = playergw.event
        GROUP BY member, memberid
    ),
    top_5_members AS (
        SELECT memberid
        FROM member_points
        WHERE memberid NOT IN ({mydivx_team_20262027}, {mydiv1_team_20262027})
        ORDER BY total_points DESC
        LIMIT 5
    ),
    my_team AS (
        SELECT DISTINCT element
        FROM divx_picks_{current_season}
        WHERE memberid = {divx_team_code}
            AND gw = {current_gw}
    ),
    latest_predictions AS (
        SELECT *
        FROM predictions
        LEFT JOIN player_status ON predictions.element = player_status.id
        WHERE prediction_timestamp = (
            SELECT MAX(prediction_timestamp)
            FROM predictions
        )
    ),
    current_divx_picks AS (
        SELECT element, SUM(multiplier) AS total_multiplier
        FROM divx_picks_{current_season}
        WHERE divx_picks_{current_season}.gw = {current_gw}
            AND memberid IN (SELECT memberid FROM top_5_members)
        GROUP BY element
    ),
    player_ownership AS (
        SELECT
            CASE WHEN mt.element IS NOT NULL THEN 1 ELSE 0 END AS in_my_team,
            p.element,
            p.player_name_id,
            p.status,
            p.now_cost / 10.0 AS value,
            p.position,
            p.team_name,
            p.event,
            p.xpoints,
            COALESCE(cdx.total_multiplier, 0)::DECIMAL / NULLIF((SELECT COUNT(*) FROM top_5_members), 0) AS ownership_ratio
        FROM latest_predictions p
        LEFT JOIN current_divx_picks cdx
            ON cdx.element = p.element
        LEFT JOIN my_team mt
            ON mt.element = p.element
        WHERE p.status IN ('a', 'd')
    ),
    aggregated_stats AS (
        SELECT 
            in_my_team,
            element, 
            player_name_id,
            status,
            value, 
            position, 
            team_name,
            SUM(xpoints) AS sum_xpoints,
            MAX(ownership_ratio) * 100.0 AS ownership_percentage,
            MAX(ownership_ratio) AS ownership_ratio,
            -- Points gained relative to top 5 if owned (0 when 100% owned, full xP when 0% owned)
            SUM(xpoints) * (1.0 - MAX(ownership_ratio)) AS differential_upside,
            -- Relative delta against rivals (-xP * O if not owned, xP * (1 - O) if owned)
            SUM(xpoints) * (in_my_team - MAX(ownership_ratio)) AS net_rank_swing
        FROM player_ownership
        WHERE event BETWEEN {current_gw + 1} AND 38
        GROUP BY in_my_team, element, player_name_id, status, value, position, team_name
    )
    SELECT 
        *,
        -- Scaled blend between raw total points and differential gain potential
        sum_xpoints * ({alpha} + (1.0 - {alpha}) * (1.0 - ownership_ratio)) AS blended_score
    FROM aggregated_stats
""")

print(f"BEST RANK MOVERS! Top 15 differential picks in my team for GW{current_gw + 1} - GW38:")
in_my_team_filter = ownership_weighted_predictions['in_my_team'] == 1
ownership_weighted_predictions[in_my_team_filter].sort_values(by='net_rank_swing', ascending=False)

BEST RANK MOVERS! Top 15 differential picks in my team for GW12 - GW38:


,in_my_team,element,player_name_id,status,value,position,team_name,sum_xpoints,ownership_percentage,ownership_ratio,differential_upside,net_rank_swing,blended_score
487,1,79.0,Francisco Evanilson de Lima Barbosa,a,6.6,FWD,Bournemouth,131.170282,0.0,0.0,131.170282,131.170282,131.170282
492,1,346.0,Dominic Calvert-Lewin,a,6.0,FWD,Leeds,119.123676,0.0,0.0,119.123676,119.123676,119.123676
496,1,480.0,Morgan Gibbs-White,a,7.8,MID,Nott'm Forest,144.636510,20.0,0.2,115.709208,115.709208,130.172859
493,1,366.0,Florian Wirtz,a,7.3,MID,Liverpool,142.673134,20.0,0.2,114.138507,114.138507,128.405820
495,1,427.0,Bryan Mbeumo,a,7.8,MID,Man Utd,105.785353,0.0,0.0,105.785353,105.785353,105.785353
489,1,173.0,Bobby Thomas,a,4.0,DEF,Coventry City,99.291778,0.0,0.0,99.291778,99.291778,99.291778
486,1,60.0,James Hill,a,5.6,DEF,Bournemouth,123.237114,20.0,0.2,98.589691,98.589691,110.913403
497,1,497.0,Martin Dubravka,a,3.9,GK,Spurs,96.694951,0.0,0.0,96.694951,96.694951,96.694951
490,1,175.0,Milan van Ewijk,a,3.9,DEF,Coventry City,93.241631,0.0,0.0,93.241631,93.241631,93.241631
488,1,154.0,Cole Palmer,a,9.8,MID,Chelsea,120.716532,40.0,0.4,72.429919,72.429919,96.573226


In [9]:
print(f"DONT MISS OUT! Most owned points scorers for GW{current_gw + 1} - GW38:")
ownership_weighted_predictions.sort_values(by='net_rank_swing', ascending=True).head(20)

DONT MISS OUT! Most owned points scorers for GW12 - GW38:


,in_my_team,element,player_name_id,status,value,position,team_name,sum_xpoints,ownership_percentage,ownership_ratio,differential_upside,net_rank_swing,blended_score
111,0,166.0,Nicolas Jackson,a,7.2,FWD,Aston Villa,131.530109,120.0,1.2,-26.306022,-157.836131,52.612044
494,1,411.0,Erling Haaland,a,15.8,FWD,Man City,157.132876,180.0,1.8,-125.706301,-125.706301,15.713288
82,0,115.0,Maxim De Cuyper,a,5.5,DEF,Brighton,112.984772,100.0,1.0,0.000000,-112.984772,56.492386
25,0,40.0,Morgan Rogers,a,8.3,MID,Chelsea,120.191563,80.0,0.8,24.038313,-96.153251,72.114938
88,0,124.0,Pascal Groß,a,6.3,MID,Brighton,150.702716,60.0,0.6,60.281086,-90.421629,105.491901
259,0,379.0,Alexander Isak,a,9.3,FWD,Liverpool,134.100505,60.0,0.6,53.640202,-80.460303,93.870353
76,0,109.0,Bart Verbruggen,a,4.7,GK,Brighton,98.102047,40.0,0.4,58.861228,-39.240819,78.481637
79,0,112.0,Jan Paul van Hecke,a,5.2,DEF,Spurs,96.307990,40.0,0.4,57.784794,-38.523196,77.046392
11,0,15.0,Martin Ødegaard,a,6.7,MID,Arsenal,149.117797,20.0,0.2,119.294238,-29.823559,134.206017
291,0,426.0,Bruno Borges Fernandes,a,11.6,MID,Man Utd,144.341618,20.0,0.2,115.473294,-28.868324,129.907456


In [10]:
print(f"Best differential picks not in my team for GW{current_gw + 1} - GW38:")
not_in_my_team_filter = ownership_weighted_predictions['in_my_team'] == 0
ownership_weighted_predictions[not_in_my_team_filter].sort_values(by='differential_upside', ascending=False).head(20)

Best differential picks not in my team for GW12 - GW38:


,in_my_team,element,player_name_id,status,value,position,team_name,sum_xpoints,ownership_percentage,ownership_ratio,differential_upside,net_rank_swing,blended_score
7,0,10.0,Benjamin White,a,5.4,DEF,Arsenal,132.856082,0.0,0.0,132.856082,0.000000,132.856082
199,0,295.0,Oli McBurnie,a,5.5,FWD,Hull City,129.551860,0.0,0.0,129.551860,0.000000,129.551860
9,0,13.0,Declan Rice,a,7.5,MID,Arsenal,129.221924,0.0,0.0,129.221924,0.000000,129.221924
436,0,627.0,Bradley Barcola,a,8.1,MID,Liverpool,128.988207,0.0,0.0,128.988207,0.000000,128.988207
12,0,16.0,Noni Madueke,a,6.2,MID,Arsenal,128.319457,0.0,0.0,128.319457,0.000000,128.319457
4,0,6.0,William Saliba,a,5.9,DEF,Arsenal,127.900617,0.0,0.0,127.900617,0.000000,127.900617
276,0,400.0,Jérémy Doku,a,7.5,MID,Man City,126.384567,0.0,0.0,126.384567,0.000000,126.384567
120,0,183.0,Jack Rudoni,a,4.9,MID,Coventry City,126.329638,0.0,0.0,126.329638,0.000000,126.329638
44,0,67.0,Rayan Vitor Simplício Rocha,a,6.3,MID,Bournemouth,126.021714,0.0,0.0,126.021714,0.000000,126.021714
8,0,11.0,Cristhian Mosquera,a,5.1,DEF,Arsenal,123.634039,0.0,0.0,123.634039,0.000000,123.634039


In [11]:
# print(f"BLENDED SCORE: POINTS & RANK MOVERS! Top 15 differential picks in my team for GW{current_gw + 1} - GW38:")
# ownership_weighted_predictions.sort_values(by='blended_score', ascending=False).head(20)

In [12]:
# print(f"\nTop 20 predicted xPoints between {current_gw+1} and {current_gw+5}:")
# next5gw_predictions = db_creator.run_sql(f"""
#     WITH predictions_status AS (
#     SELECT *
#     FROM predictions
#     LEFT JOIN player_status ON predictions.element = player_status.id
#     )
#     SELECT
#         player_name_id,
#         status,
#         position,
#         team_name,
#         value/10 as value,
#         ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
#         ROUND(SUM(xpoints),2) AS total_xpoints,
#         ROUND(AVG(mins_pred),1) AS avg_mins_pred,
#         ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
#         ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
#         ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
#     FROM predictions_status
#     WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
#         AND status IN ('a', 'd')
#         -- AND position = 'FWD'
#         -- AND value <= 45
#         -- AND team_name = 'Crystal Palace'
#     GROUP BY player_name_id, status, position, team_name, value
#     ORDER BY total_xpoints DESC
# """).dropna(subset=['value', 'total_xpoints'])
# next5gw_predictions.head(20)


In [13]:
from FPL_Optimizer import FPLOptimizer

In [14]:
print(f"\nPredicted Ownership Weighted xPoints between {current_gw+1} and {current_gw+5}:")
if __name__ == "__main__":
    custom_columns = {
        'player_name': 'player_name_id',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'blended_score'
    }
    
    optimizer = FPLOptimizer(
        df=ownership_weighted_predictions,
        budget=101.0,
        column_mapping=custom_columns
    )
    optimizer.optimize()
    
    # Get results as DataFrames
    squad = optimizer.get_squad_dataframe()
    starting_xi = optimizer.get_starting_xi_dataframe()


Predicted Ownership Weighted xPoints between 12 and 16:
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £101.0m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 3-5-2
Total Squad Cost: £95.4m / £101.0m
Total Weighted Expected Points: 1459.42
Remaining Budget: £5.6m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Dan Bentley                    Coventry City £4.0   117.54 
 ✅        Ben Wilson                     Coventry City £4.4   117.54 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Jeremy Jacquet                 Liverpool    £5.1   117.80 
 🪑     

In [15]:
print(f"\nPredicted xPoints between {current_gw+1} and {current_gw+5}:")
next5gw_predictions = db_creator.run_sql(f"""
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        player_name_id,
        status,
        position,
        team_name,
        now_cost / 10.0 AS value,
        ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
        ROUND(SUM(xpoints),2) AS total_xpoints,
        ROUND(AVG(mins_pred),1) AS avg_mins_pred,
        ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
        ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
        ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
    FROM predictions_status
    WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
        AND status IN ('a', 'd')
        -- AND position = 'FWD'
        -- AND value <= 45
        -- AND team_name = 'Crystal Palace'
    GROUP BY player_name_id, status, position, team_name, now_cost
    ORDER BY total_xpoints DESC
""").dropna(subset=['value', 'total_xpoints'])

# Usage
if __name__ == "__main__":
    custom_columns = {
        'player_name': 'player_name_id',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'total_xpoints'
    }
    
    optimizer = FPLOptimizer(
        df=next5gw_predictions,
        budget=101.0,
        column_mapping=custom_columns
    )
    optimizer.optimize()
    
    # Get results as DataFrames
    squad = optimizer.get_squad_dataframe()
    starting_xi = optimizer.get_starting_xi_dataframe()


Predicted xPoints between 12 and 16:
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £101.0m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 3-5-2
Total Squad Cost: £100.9m / £101.0m
Total Weighted Expected Points: 291.93
Remaining Budget: £0.1m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Dan Bentley                    Coventry City £4.0    22.10 
 ✅        Ben Wilson                     Coventry City £4.4    22.10 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Victor Lindelöf                Aston Villa  £4.5    20.98 
 🪑        Luke O'Nien     

In [16]:
remaining_gw_predictions = db_creator.run_sql(f"""
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        event,
        player_name_id,
        status,
        position,
        team_name,
        opp_team_name,
        now_cost / 10.0 AS value,
        xpoints
    FROM predictions_status
    WHERE event BETWEEN {current_gw+1} AND 38
        AND status IN ('a', 'd')
    ORDER BY xpoints DESC
""")
print("Best week to triple captain:")
remaining_gw_predictions.head(10)

Best week to triple captain:


,event,player_name_id,status,position,team_name,opp_team_name,value,xpoints
0,36.0,Pascal Groß,a,MID,Brighton,Crystal Palace,6.3,6.309381
1,30.0,Pascal Groß,a,MID,Brighton,Coventry City,6.3,6.297547
2,16.0,Pascal Groß,a,MID,Brighton,Ipswich Town,6.3,6.286257
3,27.0,Pascal Groß,a,MID,Brighton,Ipswich Town,6.3,6.286257
4,32.0,Erling Haaland,a,FWD,Man City,Crystal Palace,15.8,6.184004
5,36.0,Erling Haaland,a,FWD,Man City,Ipswich Town,15.8,6.160163
6,29.0,Erling Haaland,a,FWD,Man City,Coventry City,15.8,6.139509
7,33.0,Bruno Borges Fernandes,a,MID,Man Utd,Crystal Palace,11.6,6.114833
8,15.0,Bruno Borges Fernandes,a,MID,Man Utd,Crystal Palace,11.6,6.114833
9,38.0,Erling Haaland,a,FWD,Man City,Sunderland,15.8,6.085605


In [17]:
print(f"\nOptimizing team selection for GW{current_gw+1} to GW38...")

if __name__ == "__main__":
    custom_columns = {
        'player_name': 'player_name_id',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'xpoints'
    }
    all_teams =pd.DataFrame()
    for event in range(current_gw+1, 19): #change to 39 for 2nd half of season
        print(f"\nOptimizing team selection for GW{event}...")
        try:
            optimizer = FPLOptimizer(
                df=remaining_gw_predictions[remaining_gw_predictions['event']==event],
                budget=100.0,
                column_mapping=custom_columns
            )
            optimizer.optimize()
        
            # Get results as DataFrames
            # squad = optimizer.get_squad_dataframe()
            starting_xi = optimizer.get_starting_xi_dataframe()
            # result = optimizer.calculate_bench_boost_xpoints()
            starting_xi['event'] = event
            all_teams = pd.concat([all_teams, starting_xi], ignore_index=True)
        except Exception as e:
            print(f"Error optimizing for GW{event}: {e}")


Optimizing team selection for GW12 to GW38...

Optimizing team selection for GW12...
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £100.0m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 3-5-2
Total Squad Cost: £100.0m / £100.0m
Total Weighted Expected Points: 57.51
Remaining Budget: £0.0m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Dino Kaiser                    Fulham       £4.0     4.46 
 ✅        Dan Bentley                    Coventry City £4.0     4.47 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        Nobel Mendy                    Hull Cit

In [18]:
freehit_points = all_teams.pivot_table(index='event', values='xpoints', aggfunc='sum').sort_values(by='xpoints', ascending=False)
best_week = freehit_points.head(1)['xpoints'].index.astype(int).values[0]
print(f"Best week to freehit: {best_week}, points: {round(all_teams[all_teams['event']==best_week]['xpoints'].sum(),1)}")
all_teams[all_teams['event']==best_week].sort_values(by=['position','xpoints'], ascending=False)

Best week to freehit: 14, points: 58.4


,event,player_name_id,status,position,team_name,opp_team_name,value,xpoints
22,14,Bruno Borges Fernandes,a,MID,Man Utd,Coventry City,11.6,6.052367
24,14,Martin Ødegaard,a,MID,Arsenal,Spurs,6.7,5.487597
25,14,Pascal Groß,a,MID,Brighton,Nott'm Forest,6.3,5.451750
27,14,Florian Wirtz,a,MID,Liverpool,Chelsea,7.3,5.339989
30,14,Marcus Tavernier,a,MID,Bournemouth,Hull City,6.7,5.065041
32,14,Dan Bentley,a,GK,Coventry City,Man Utd,4.0,4.491360
23,14,Erling Haaland,a,FWD,Man City,Brentford,15.8,5.813044
29,14,Nicolas Jackson,a,FWD,Aston Villa,Crystal Palace,7.2,5.180938
26,14,Ezri Konsa Ngoyo,a,DEF,Arsenal,Spurs,4.9,5.411058
28,14,Benjamin White,a,DEF,Arsenal,Spurs,5.4,5.247810


In [19]:
# finish=time.perf_counter()
# print(f'Finished in {round(((finish-start)/60),2)} minute(s)') 
# #Finished in 0.01 minute(s)

In [20]:
# Notification when file has finished running
from plyer import notification

notification.notify(
    title='Script Complete',
    message='Your file has finished running!',
    app_name='Jupyter Notebook',
    timeout=20  # notification stays for 20 seconds
)